# 02 · Preprocessing, Feature Engineering & SMOTE
Shows each step on its own. In production the same steps run inside one `imblearn` pipeline (see `src/train.py`) so there is no leakage.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
from collections import Counter
from imblearn.over_sampling import SMOTENC
from src.config import DEFAULT_RAW_DATA, TARGET, SMOTE_SAMPLING_STRATEGY
from src.preprocessing import load_dataset, clean_dataset, split_data, build_preprocessing_pipeline, n_output_features
from src.feature_engineering import FeatureEngineer

df, _ = clean_dataset(load_dataset(DEFAULT_RAW_DATA))

## 1 · Feature engineering (derived only from existing columns)

In [ ]:
engineered = FeatureEngineer().transform(df.drop(columns=[TARGET]).head(5))
engineered.T

## 2 · Stratified split FIRST (before any resampling)

In [ ]:
X_train, X_test, y_train, y_test = split_data(df)
print("train fraud rate:", round(y_train.mean(), 4), "| test fraud rate:", round(y_test.mean(), 4))

## 3 · Fit preprocessing on TRAIN only (impute → scale → one-hot)

In [ ]:
pipe = build_preprocessing_pipeline().fit(X_train)
Xtr, Xte = pipe.transform(X_train), pipe.transform(X_test)
names = pipe.named_steps["prep"].get_feature_names_out()
print(Xtr.shape, Xte.shape, "NaNs:", int(np.isnan(Xtr).sum()))
list(names)

## 4 · SMOTENC on TRAIN only
SMOTENC = SMOTE for mixed continuous/categorical data. The test set is left untouched and imbalanced.

In [ ]:
n_num, n_total = n_output_features()
smote = SMOTENC(categorical_features=list(range(n_num, n_total)), sampling_strategy=SMOTE_SAMPLING_STRATEGY, random_state=42)
Xres, yres = smote.fit_resample(Xtr, y_train)
print("before:", Counter(y_train), "\nafter :", Counter(yres), "\ntest  :", Counter(y_test), "(unchanged)")

In `src/train.py` this step lives *inside* the cross-validation pipeline, so each CV fold resamples only its own training portion.